In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. SIMULATION PARAMETERS
# ==========================================
NUM_NODES = 1000        # Number of substations
TOLERANCE = 0.10        # Buffer capacity (20% extra)
NOISE = 1.9             # Stress added per tick
SIMULATION_STEPS = 8000 # How many ticks to run
RADIUS = 0.05           # Connection distance for nodes

# --- NEW: Print Input Parameters ---
print("-" * 30)
print("POWER GRID SIMULATION INPUTS")
print("-" * 30)
print(f"Nodes (Substations): {NUM_NODES}")
print(f"Radius (Connectivity): {RADIUS}")
print(f"Tolerance (Buffer):   {TOLERANCE * 100}%")
print(f"Noise (Stress Rate):  {NOISE}")
print(f"Ticks (Steps):        {SIMULATION_STEPS}")
print("-" * 30)

# ==========================================
# 2. BUILD THE NETWORK
# ==========================================
G = nx.random_geometric_graph(NUM_NODES, radius=RADIUS)

for node in G.nodes():
    initial_load = np.random.uniform(0, 10)
    G.nodes[node]['load'] = initial_load
    G.nodes[node]['capacity'] = initial_load * (1 + TOLERANCE)

avalanche_data = []

# ==========================================
# 3. RUN THE SIMULATION
# ==========================================
print(f"Running simulation...")

for step in range(SIMULATION_STEPS):
    for node in G.nodes():
        G.nodes[node]['load'] += (NOISE / 10.0)
        G.nodes[node]['tripped'] = False

    current_avalanche_size = 0

    while True:
        overloaded_nodes = [
            n for n, data in G.nodes(data=True)
            if data['load'] > data['capacity'] and not data['tripped']
        ]

        if not overloaded_nodes:
            break

        for node in overloaded_nodes:
            overload_amount = G.nodes[node]['load']
            G.nodes[node]['load'] = 0
            G.nodes[node]['tripped'] = True
            current_avalanche_size += 1

            active_neighbors = [nbr for nbr in G.neighbors(node) if not G.nodes[nbr]['tripped']]

            if len(active_neighbors) > 0:
                spillover = overload_amount / len(active_neighbors)
                for neighbor in active_neighbors:
                    G.nodes[neighbor]['load'] += spillover

    if current_avalanche_size > 0:
        avalanche_data.append(current_avalanche_size)

print(f"Simulation complete! Recorded {len(avalanche_data)} avalanches.")

# ==========================================
# 4. PLOT THE LOG-LOG CCDF
# ==========================================
if len(avalanche_data) > 0:
    sorted_sizes = np.sort(avalanche_data)[::-1]
    probabilities = np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes)

    plt.figure(figsize=(8, 6))
    plt.loglog(sorted_sizes, probabilities, marker='o', linestyle='none', color='#ff7f0e', alpha=0.6)

    plt.title(f'Power Grid Cascade (N={NUM_NODES}, R={RADIUS})')
    plt.xlabel('Avalanche Size (Number of failed nodes)')
    plt.ylabel('Probability P(Size > x)')
    plt.grid(True, which="both", ls="--", alpha=0.3)
    plt.show()
else:
    print("No avalanches occurred. Try increasing NOISE or lowering TOLERANCE.")

------------------------------
POWER GRID SIMULATION INPUTS
------------------------------
Nodes (Substations): 1000
Radius (Connectivity): 0.05
Tolerance (Buffer):   10.0%
Noise (Stress Rate):  1.9
Ticks (Steps):        8000
------------------------------
Running simulation...
